In [1]:
# ══════════════════════════════════════════════════════
# PARAMÈTRES GLOBAUX — modifier ici uniquement
# ══════════════════════════════════════════════════════

TAILLE_CELLULE = 1.0    # taille des mailles en degrés (ex: 0.5, 1.0, 2.0)
ANNEE_DEBUT    = 1970   # début de la période
ANNEE_FIN      = 2021   # fin de la période
ANNEE_SPLIT    = 2015   # séparation train / test

FICHIERS_GTD = [
    "../data/globalterrorismdb_0522dist.xlsx",
    "../data/globalterrorismdb_2021Jan-June_1222dist.xlsx",
]

COLONNES_UTILES = [
    "eventid",
    "iyear", "imonth", "iday",
    "country_txt", "region_txt",
    "latitude", "longitude",
    "attacktype1_txt",
    "targtype1_txt",
    "gname",
    "weaptype1_txt",
    "nkill", "nwound",
    "success", "suicide"
]

print("Paramètres chargés !")
print(f"  Taille cellule : {TAILLE_CELLULE}°")
print(f"  Période        : {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"  Train/Test     : ≤{ANNEE_SPLIT} / >{ANNEE_SPLIT}")

Paramètres chargés !
  Taille cellule : 1.0°
  Période        : 1970–2021
  Train/Test     : ≤2015 / >2015


In [2]:
import pandas as pd
import numpy as np

# ── CHARGEMENT ET FUSION DES FICHIERS GTD ──
print("Chargement des données...")

dfs = []
for fichier in FICHIERS_GTD:
    df_temp = pd.read_excel(fichier, engine="openpyxl")
    cols = [c for c in COLONNES_UTILES if c in df_temp.columns]
    dfs.append(df_temp[cols])
    print(f"  {fichier.split('/')[-1]} — {len(df_temp):,} lignes")

df = pd.concat(dfs, ignore_index=True)

# ── NETTOYAGE ──
df = df[df["iyear"].between(ANNEE_DEBUT, ANNEE_FIN)]
df = df.dropna(subset=["latitude", "longitude"])
df["nkill"]  = df["nkill"].fillna(0).clip(lower=0)
df["nwound"] = df["nwound"].fillna(0).clip(lower=0)

print(f"\nDonnées chargées et nettoyées !")
print(f"  Lignes totales : {len(df):,}")
print(f"  Période        : {df['iyear'].min()}–{df['iyear'].max()}")

Chargement des données...
  globalterrorismdb_0522dist.xlsx — 209,706 lignes
  globalterrorismdb_2021Jan-June_1222dist.xlsx — 4,960 lignes

Données chargées et nettoyées !
  Lignes totales : 209,939
  Période        : 1970–2021


In [3]:
# ── DÉCOUPAGE EN CELLULES ──
df["cell_lat"] = (df["latitude"]  // TAILLE_CELLULE) * TAILLE_CELLULE
df["cell_lon"] = (df["longitude"] // TAILLE_CELLULE) * TAILLE_CELLULE
df["cell_id"]  = df["cell_lat"].astype(str) + "_" + df["cell_lon"].astype(str)

print(f"Cellules distinctes : {df['cell_id'].nunique():,}")

# ── AGRÉGATION PAR CELLULE ET ANNÉE ──
grid = df.groupby(["cell_id", "cell_lat", "cell_lon", "iyear"]).agg(
    nb_incidents = ("eventid", "count"),
    nb_morts     = ("nkill", "sum"),
    nb_blesses   = ("nwound", "sum"),
).reset_index()

# ── AJOUT DES ZÉROS (cellules sans attentat) ──
toutes_les_annees   = range(ANNEE_DEBUT, ANNEE_FIN + 1)
toutes_les_cellules = df[["cell_id", "cell_lat", "cell_lon"]].drop_duplicates()

index_complet = toutes_les_cellules.merge(
    pd.DataFrame({"iyear": toutes_les_annees}), how="cross"
)

grid = index_complet.merge(grid, on=["cell_id", "cell_lat", "cell_lon", "iyear"], how="left")
grid[["nb_incidents", "nb_morts", "nb_blesses"]] = grid[["nb_incidents", "nb_morts", "nb_blesses"]].fillna(0)

# ── VARIABLE CIBLE ──
# Classification : présence ou absence d'attentat
grid["target_clf"] = (grid["nb_incidents"] > 0).astype(int)
# Régression : nombre d'attentats
grid["target_reg"] = grid["nb_incidents"]

# ── FEATURE ENGINEERING ──
grid = grid.sort_values(["cell_id", "iyear"]).reset_index(drop=True).copy()

grid["incidents_annee_precedente"] = (
    grid.groupby("cell_id")["nb_incidents"].shift(1).fillna(0)
)
grid["moyenne_3ans"] = (
    grid.groupby("cell_id")["nb_incidents"]
    .shift(1)
    .groupby(grid["cell_id"])
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
    .fillna(0)
)
grid["moyenne_5ans"] = (
    grid.groupby("cell_id")["nb_incidents"]
    .shift(1)
    .groupby(grid["cell_id"])
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .fillna(0)
)
grid["nb_annees_actives"] = (
    grid.groupby("cell_id")["target_clf"]
    .shift(1)
    .fillna(0)
    .groupby(grid["cell_id"])
    .transform(lambda x: x.expanding().sum())
)

print(f"Grille complète : {len(grid):,} lignes")
print(f"Taux de cellules actives : {grid['target_clf'].mean()*100:.1f}%")

Cellules distinctes : 4,085
Grille complète : 212,420 lignes
Taux de cellules actives : 12.6%


In [4]:
# ── FEATURES ET SÉPARATION TRAIN/TEST ──

FEATURES = [
    "cell_lat",
    "cell_lon",
    "iyear",
    "incidents_annee_precedente",
    "moyenne_3ans",
    "moyenne_5ans",
    "nb_annees_actives",
]

train = grid[grid["iyear"] <= ANNEE_SPLIT]
test  = grid[grid["iyear"] >  ANNEE_SPLIT]

# Classification
X_train = train[FEATURES]
y_train_clf = train["target_clf"]
X_test  = test[FEATURES]
y_test_clf  = test["target_clf"]

# Régression
y_train_reg = train["target_reg"]
y_test_reg  = test["target_reg"]

print(f"Train : {len(X_train):,} lignes ({train['iyear'].min()}–{train['iyear'].max()})")
print(f"Test  : {len(X_test):,} lignes ({test['iyear'].min()}–{test['iyear'].max()})")
print(f"\nTaux positifs train : {y_train_clf.mean()*100:.1f}%")
print(f"Taux positifs test  : {y_test_clf.mean()*100:.1f}%")

Train : 187,910 lignes (1970–2015)
Test  : 24,510 lignes (2016–2021)

Taux positifs train : 11.0%
Taux positifs test  : 25.4%


In [5]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# ── NORMALISATION ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

resultats = {}

# ── 1. RÉGRESSION LINÉAIRE ──
lr_reg = LinearRegression()
lr_reg.fit(X_train_scaled, y_train_reg)
y_pred_linreg = lr_reg.predict(X_test_scaled).clip(0)

resultats["Régression Linéaire"] = {
    "RMSE" : np.sqrt(mean_squared_error(y_test_reg, y_pred_linreg)),
    "MAE"  : mean_absolute_error(y_test_reg, y_pred_linreg),
    "R²"   : r2_score(y_test_reg, y_pred_linreg),
}
print("=== RÉGRESSION LINÉAIRE ===")
for k, v in resultats["Régression Linéaire"].items():
    print(f"  {k} : {v:.4f}")

# ── 2. RÉGRESSION LOGISTIQUE ──
lr_clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_clf.fit(X_train_scaled, y_train_clf)
y_pred_lr  = lr_clf.predict(X_test_scaled)
y_proba_lr = lr_clf.predict_proba(X_test_scaled)[:, 1]

print("\n=== RÉGRESSION LOGISTIQUE ===")
print(classification_report(y_test_clf, y_pred_lr))
print(f"  AUC-ROC : {roc_auc_score(y_test_clf, y_proba_lr):.4f}")
resultats["Régression Logistique"] = {"AUC-ROC": roc_auc_score(y_test_clf, y_proba_lr)}

# ── 3. RANDOM FOREST ──
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train_clf)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("\n=== RANDOM FOREST ===")
print(classification_report(y_test_clf, y_pred_rf))
print(f"  AUC-ROC : {roc_auc_score(y_test_clf, y_proba_rf):.4f}")
resultats["Random Forest"] = {"AUC-ROC": roc_auc_score(y_test_clf, y_proba_rf)}

# ── 4. XGBOOST ──
xgb = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=len(y_train_clf[y_train_clf==0]) / len(y_train_clf[y_train_clf==1]),
    random_state=42, n_jobs=-1, eval_metric="logloss", verbosity=0
)
xgb.fit(X_train, y_train_clf)
y_pred_xgb  = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("\n=== XGBOOST ===")
print(classification_report(y_test_clf, y_pred_xgb))
print(f"  AUC-ROC : {roc_auc_score(y_test_clf, y_proba_xgb):.4f}")
resultats["XGBoost"] = {"AUC-ROC": roc_auc_score(y_test_clf, y_proba_xgb)}

print("\n=== RÉCAPITULATIF AUC-ROC ===")
for modele, scores in resultats.items():
    if "AUC-ROC" in scores:
        print(f"  {modele:25s} : {scores['AUC-ROC']:.4f}")

=== RÉGRESSION LINÉAIRE ===
  RMSE : 11.5996
  MAE : 2.0011
  R² : 0.5457

=== RÉGRESSION LOGISTIQUE ===
              precision    recall  f1-score   support

           0       0.91      0.63      0.74     18293
           1       0.43      0.82      0.56      6217

    accuracy                           0.67     24510
   macro avg       0.67      0.72      0.65     24510
weighted avg       0.79      0.67      0.70     24510

  AUC-ROC : 0.8140

=== RANDOM FOREST ===
              precision    recall  f1-score   support

           0       0.87      0.90      0.89     18293
           1       0.68      0.61      0.64      6217

    accuracy                           0.83     24510
   macro avg       0.78      0.76      0.77     24510
weighted avg       0.82      0.83      0.83     24510

  AUC-ROC : 0.8366

=== XGBOOST ===
              precision    recall  f1-score   support

           0       0.93      0.57      0.71     18293
           1       0.41      0.87      0.56      6217
